# 7. Shift-share (Bartik) instrument

## What the instrument is for

Every estimate so far is a correlation. Even the within-pair specification in notebook 5 —
which removes the firm's permanent technology profile and the national trend in each
technology — leaves one threat standing: a group that decides to push a technology may
raise its network's patenting *and* its Italian affiliate's patenting in the same years, for
the same reason. Firm × year FE absorb firm-level shocks, subclass × year FE absorb
technology-level shocks, but a shock specific to *this firm in this technology this year* is
absorbed by neither. That is what the instrument is meant to break.

## How a shift-share works, briefly

A Bartik instrument predicts a unit's exposure from two pieces that are assumed to be
unrelated to the outcome once combined: a **share**, fixed in a pre-period, saying how much
the unit is tied to each category; and a **shift**, the aggregate movement in that category,
common to everyone. The product gives a predicted exposure that moves for reasons outside
the unit's own control. Goldsmith-Pinkham, Sorkin & Swift (2020) show that identification
rests on the **shares** being exogenous — the shifts are doing the timing, the shares are
doing the identifying.

Our treatment is already indexed by technology, so the construction is simpler than the
classic regional version. For firm $i$, subclass $s$, year $t$:

$$w_{is} \;=\; \frac{\text{network's pre-2015 families in } s}{\text{network's pre-2015 families, all subclasses}}, \qquad \sum_s w_{is} = 1$$

$$G_{st} \;=\; \text{cumulative families in } s \text{ filed before } t \text{ by all groups other than } i\text{'s}$$

$$Z_{ist} \;=\; \log\!\left(1 + w_{is}\,G_{st}\right)$$

$w_{is}$ is the share — where the network's knowledge base sat before the window opened.
$G_{st}$ is the shift — how much the technology grew nationally, computed **leave-one-out**
so that firm $i$'s own group never contributes to its own instrument. No new data: $G_{st}$
is built from the existing sample.

## Which specification we instrument, and why

The pair-FE specification, not the between-technology one. That looks backwards — the
pair-FE arm is the one that is already null and thin — but the between-technology
specification cannot be instrumented validly here.

(Both are panels. The distinction is which contrast identifies β: `firm_year + sub_year`
compares different subclasses within the same firm-year, while adding the pair FE moves
identification to the same firm-subclass over time.) "Cross-sectional" here means the contrast is across technologies within a firm-year, not
that the data are a cross-section — the panel is the same in both specifications.

Identification comes from the shares. Notebook 6 established that the network's pre-period
technology composition is strongly correlated with the affiliate's own: pairs the network
enters were already patenting at 0.338 against 0.181 for pairs it never enters. So $w_{is}$
is correlated with the outcome directly, and under `firm_year + sub_year` alone the
exclusion restriction fails at the first step — the instrument's identifying variation *is*
the sorting we are trying to remove.

Adding the pair FE absorbs $w_{is}$ entirely. What remains is $w_{is}$ interacted with
deviations in $G_{st}$, which varies within a firm-subclass over time, and the level
correlation between shares and the outcome is gone. The subclass × year FE absorb the common
effect of $G_{st}$, so the surviving variation is the *differential* response by share.

The estimating equations, on the 4-character risk set:

$$\textbf{First stage:}\quad x_{ist} \;=\; \pi\,Z_{ist} \;+\; \alpha_{it} + \gamma_{st} + \delta_{is} \;+\; \nu_{ist}$$

$$\textbf{Second stage:}\quad y_{ist} \;=\; \beta\,\hat{x}_{ist} \;+\; \alpha_{it} + \gamma_{st} + \delta_{is} \;+\; \varepsilon_{ist}$$

with $y$ = new family filed, $x = \log(1 + \text{net\_stock})$, $\alpha_{it}$ firm × year,
$\gamma_{st}$ subclass × year, $\delta_{is}$ firm-subclass, SEs clustered by parent group. The
endogenous regressor is the continuous form rather than `net_any`: a binary endogenous
variable with a continuous instrument is awkward to interpret and the first stage behaves
badly.

**What has to hold.** Conditional on the fixed effects, a network with a larger pre-2015
share in subclass $s$ must not have an affiliate whose own patenting responds differently to
$s$'s national trend, except through the network's stock. That is a real assumption, not a
formality — it is violated if, say, groups concentrated in a technology are also better at
exploiting booms in it. It is weaker than what the cross-sectional version would require.

**What to expect.** The first stage must predict movement in `net_stock`, which happens in
390 of 2,744 pairs. Expect a modest $F$ and second-stage intervals wider than the OLS. The
notebook's purpose is to close the loop — to report the IV, its first stage, and why it is
weak — rather than to leave the shift-share as an unexamined line in the conclusion.

**Why `log_net` as the endogenous regressor.** It matches the instrument's functional form —
the first stage is $\log(1 + w_{is}G_{st})$ on $\log(1 + \text{net\_stock})$, so $\beta$
reads as the effect of a proportional increase in network stock. `net_any` is rejected
because a binary endogenous variable with a continuous instrument identifies a LATE over a
complier group we cannot characterise, and it switches in only 127 pairs. Levels are
rejected because the coefficient is unreadable at three decimals and behaved inconsistently
across variance estimators in notebooks 4 and 5.

One honest qualification: `log_net` is not cleanly continuous here. About 75% of risk-set
cells have no network stock, so the variable is a mass point at zero plus a continuous tail,
and $\beta$ blends an extensive margin (any network presence) with an intensive one (more of
it). This is the same object estimated throughout notebooks 4–6, so the IV is comparable to
the OLS, but it should not be described as purely the effect of additional network families.

**Clustering.** Parent group, not firm. In notebooks 4–6 the two were nearly identical and
firm level was adequate. Here the instrument is built from the network's pre-2015
composition and a group-level leave-one-out shift, so affiliates of the same parent receive
near-identical instrument values in a given subclass-year — the effective unit of variation
is the group. Firm-level clustering would treat those as independent draws. Firm-level and
HC1 are reported alongside for comparability with the earlier notebooks.

In [1]:
import numpy as np
import pandas as pd
import pyfixest as pf

data_dir = r"C:\Users\Popov\Documents\Studies\IMT_studies\Projects\Econ_2\Datasets"
panel = pd.read_parquet(data_dir + r"\panel_firm_subclass_year.parquet")
patents = pd.read_stata(data_dir + r"\patents.dta")

FIRM, PARENT, SUB, YEAR = "bvd_key", "parent_key", "subclass", "year"
Y, X = "patented", "net_stock"
PFIRM, PFAM, PDATE = "bvd_key", "family_identifier__simple", "priority_date"

# pre-2015 patent history, subclass level
valid = patents.dropna(subset=[PDATE, PFAM])
pre = valid[valid[PDATE].dt.year < 2015].copy()
pre["sub"] = pre["ipc_code__main"].str[:4]
pre = pre[pre["sub"] != ""]

# risk set: firm-subclass pairs the firm patented in before 2015
keep = set(map(tuple, pre.loc[pre[PFIRM].isin(set(panel[FIRM])),
                              [PFIRM, "sub"]].drop_duplicates().values))
rs = panel[[(f, s) in keep for f, s in zip(panel[FIRM], panel[SUB])]].copy()
rs["pair"] = rs[FIRM] + "|" + rs[SUB]
rs["firm_id"] = rs[FIRM].astype("category").cat.codes

print(f"{len(rs):,} cells | {rs[FIRM].nunique():,} firms | "
      f"{rs[PARENT].nunique():,} groups | {rs['pair'].nunique():,} pairs")

27,440 cells | 516 firms | 253 groups | 2,744 pairs


### Building the shares

$w_{is}$ is the share of the *network's* pre-2015 families sitting in subclass $s$ — where
the firm's Italian co-affiliates had built their knowledge base before the window opened.
Self-excluded: firm $i$'s own families never enter its own share, matching how `net_stock`
is constructed.

$$w_{is} = \frac{\text{pre-2015 families in } s \text{ owned by } i\text{'s co-affiliates}}{\text{pre-2015 families in all subclasses owned by } i\text{'s co-affiliates}}$$

In [2]:
# map each firm to its group, and pre-2015 families to (firm, subclass)
grp = panel[[FIRM, PARENT]].drop_duplicates()
pf_pre = (pre[[PFIRM, "sub", PFAM]].drop_duplicates()
            .merge(grp, left_on=PFIRM, right_on=FIRM, how="inner"))

# group totals by subclass, and each firm's own contribution
g_sub = pf_pre.groupby([PARENT, "sub"])[PFAM].nunique().rename("g_s")
f_sub = pf_pre.groupby([PARENT, FIRM, "sub"])[PFAM].nunique().rename("f_s")

shares = f_sub.reset_index().merge(g_sub.reset_index(), on=[PARENT, "sub"])
shares["net_s"] = shares["g_s"] - shares["f_s"]          # co-affiliates only

tot = shares.groupby(FIRM)["net_s"].sum().rename("net_tot")
shares = shares.merge(tot, on=FIRM)
shares["w"] = np.where(shares["net_tot"] > 0, shares["net_s"] / shares["net_tot"], 0.0)

print(f"{shares[FIRM].nunique():,} firms with shares | "
      f"{(shares.groupby(FIRM)['w'].sum().round(6) == 1).mean():.1%} sum to 1")
print(shares["w"].describe())

516 firms with shares | 45.7% sum to 1
count    3375.000000
mean        0.069926
std         0.219881
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: w, dtype: float64


Only 236 of 516 firms (45.7%) have shares summing to one. The remainder have co-affiliates
with no pre-2015 patents at all, so $w_{is} = 0$ in every subclass and their instrument is
identically zero — they contribute nothing to the first stage.

This follows from how the sample was defined: the 674-firm rule required a patenting
co-affiliate at some point in 2015–2024, not before 2015. Many groups have only one
affiliate patenting in the pre-period. The effect is to compound the thinness already
documented — the first stage runs on at most 236 firms, intersected with the ~390 pairs
where `net_stock` moves within the window.

### Building the shifts

$G_{st}$ is the cumulative count of families in subclass $s$ filed before year $t$, computed
**leave-one-out**: firm $i$'s own group never contributes to the shift it faces. Without
that exclusion the instrument would contain the very network stock it is meant to predict.

$$G^{(-g)}_{st} = \sum_{\text{groups } h \neq g}\ \#\{\text{families in } s \text{ filed before } t \text{ by } h\}$$

$$Z_{ist} = \log\!\left(1 + w_{is}\,G^{(-g(i))}_{st}\right)$$

In [3]:
# all in-window families by group, subclass, year
fam = (valid[valid[PDATE].dt.year.between(2015, 2024)]
         [[PFIRM, "ipc_code__main", PFAM, PDATE]].copy())
fam["sub"] = fam["ipc_code__main"].str[:4]
fam["yr"] = fam[PDATE].dt.year
fam = fam[fam["sub"] != ""].merge(grp, left_on=PFIRM, right_on=FIRM, how="inner")

cnt = fam.groupby([PARENT, "sub", "yr"])[PFAM].nunique().rename("n").reset_index()

# cumulative national stock by subclass-year, and each group's own cumulative contribution
tot_sy = cnt.groupby(["sub", "yr"])["n"].sum().rename("n_all").reset_index()
full = (tot_sy[["sub"]].drop_duplicates()
          .merge(pd.DataFrame({"yr": range(2015, 2025)}), how="cross")
          .merge(tot_sy, on=["sub", "yr"], how="left").fillna({"n_all": 0}))
full = full.sort_values(["sub", "yr"])
full["G_all"] = full.groupby("sub")["n_all"].cumsum().shift(0) - full["n_all"]  # before t

own = cnt.sort_values([PARENT, "sub", "yr"]).copy()
own["G_own"] = own.groupby([PARENT, "sub"])["n"].cumsum() - own["n"]

# assemble: risk set + shares + leave-one-out shift
d = (rs.merge(shares[[FIRM, "sub", "w"]], left_on=[FIRM, SUB], right_on=[FIRM, "sub"], how="left")
       .merge(full[["sub", "yr", "G_all"]], left_on=[SUB, YEAR], right_on=["sub", "yr"], how="left")
       .merge(own[[PARENT, "sub", "yr", "G_own"]],
              left_on=[PARENT, SUB, YEAR], right_on=[PARENT, "sub", "yr"], how="left"))

d["w"] = d["w"].fillna(0.0)
d[["G_all", "G_own"]] = d[["G_all", "G_own"]].fillna(0.0)
d["G_loo"] = (d["G_all"] - d["G_own"]).clip(lower=0)
d["Z"] = np.log1p(d["w"] * d["G_loo"])
d["log_net"] = np.log1p(d[X])

print(f"{len(d):,} cells | w>0 in {(d.w > 0).mean():.1%} | Z>0 in {(d.Z > 0).mean():.1%}")
chk = d.groupby("pair").agg(z=("Z", "nunique"), x=("log_net", "nunique"))
print(f"pairs with Z varying: {(chk.z > 1).sum():,} of {len(chk):,}")
print(f"  ...and log_net varying too: {((chk.z > 1) & (chk.x > 1)).sum():,}")

27,440 cells | w>0 in 25.0% | Z>0 in 20.5%
pairs with Z varying: 670 of 2,744
  ...and log_net varying too: 278


The instrument varies in 670 pairs, and in 278 of those `log_net` varies as well. That
second number is the first-stage sample: 278 of the ~390 pairs where network stock moves
within the window, so the shift-share reaches roughly 71% of the usable variation. Thin, but
enough to estimate rather than abandon.

### First stage

$$\log(1 + \text{net\_stock})_{ist} = \pi Z_{ist} + \alpha_{it} + \gamma_{st} + \delta_{is} + \nu_{ist}$$

With a single instrument the first-stage $F$ is just the squared $t$ on $\pi$. The usual
reference point is 10 (Staiger–Stock); Lee et al. (2022) argue for far higher thresholds if
you want honest 5% coverage, so treat 10 as necessary rather than sufficient.

In [4]:
fe_iv = "firm_year + sub_year + pair"

fs = {}
for lab, v in [("Firm", {"CRV1": "firm_id"}),
               ("HC1", "hetero"),
               ("Parent group", {"CRV1": PARENT})]:
    m = pf.feols(f"log_net ~ Z | {fe_iv}", data=d, vcov=v)
    t = m.tidy().iloc[0]
    fs[lab] = m
    print(f"{lab:13s} pi = {t['Estimate']:.4f} ({t['Std. Error']:.4f})  "
          f"t = {t['t value']:.2f}  F = {t['t value']**2:.1f}")

Firm          pi = 0.0333 (0.0093)  t = 3.56  F = 12.7
HC1           pi = 0.0333 (0.0058)  t = 5.73  F = 32.8
Parent group  pi = 0.0333 (0.0093)  t = 3.58  F = 12.8


c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


The first stage works. $\pi = 0.033$ (0.009), positive as expected — predicted exposure
predicts actual exposure — with $F = 12.8$ under parent-group clustering and 12.7 under
firm. Above the conventional threshold of 10, so 2SLS is estimable.

Two qualifications. First, $F \approx 13$ clears Staiger–Stock but not the modern bar: Lee
et al. (2022) show that honest 5% coverage from a conventional 2SLS interval requires $F$
around 100, and at 13 the interval needs substantial inflation. Second, the firm and
parent-group columns are nearly identical (0.0093 against 0.0093), so the concern that the
group-level construction of the instrument would require coarser clustering does not bite
empirically — the same pattern as the OLS notebooks. HC1 overstates precision as usual and
should be ignored here.

### Second stage

$$y_{ist} = \beta\,\widehat{\log(1 + \text{net\_stock})}_{ist} + \alpha_{it} + \gamma_{st} + \delta_{is} + \varepsilon_{ist}$$

Reported against the OLS on the same sample and specification, so the comparison is
like-for-like. A large gap between the two would say the within-pair OLS was biased; a
similar coefficient with wider errors says the endogeneity the instrument targets was not
what was driving the null.

In [5]:
iv, ols = {}, {}
for lab, v in [("Firm", {"CRV1": "firm_id"}),
               ("HC1", "hetero"),
               ("Parent group", {"CRV1": PARENT})]:
    iv[lab]  = pf.feols(f"{Y} ~ 1 | {fe_iv} | log_net ~ Z", data=d, vcov=v)
    ols[lab] = pf.feols(f"{Y} ~ log_net | {fe_iv}", data=d, vcov=v)

for lab in iv:
    t, o = iv[lab].tidy().iloc[0], ols[lab].tidy().iloc[0]
    print(f"{lab:13s} IV {t['Estimate']:+.4f} ({t['Std. Error']:.4f})   "
          f"OLS {o['Estimate']:+.4f} ({o['Std. Error']:.4f})")

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


Firm          IV -1.1398 (0.4457)   OLS -0.0112 (0.0239)
HC1           IV -1.1398 (0.2911)   OLS -0.0112 (0.0177)
Parent group  IV -1.1398 (0.4957)   OLS -0.0112 (0.0251)


c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


$\beta_{IV} = -1.14$ is not a plausible parameter. The outcome is a probability bounded in
$[0,1]$, and `log_net` runs from 0.69 to 7.0 among exposed cells, so the estimate implies a
204 percentage point fall across its interquartile range alone. When an IV lands outside the
feasible range of the dependent variable, the cause is the specification, not the economics.

Two checks separate the candidate explanations.

**The reduced form** regresses the outcome directly on the instrument. In the
just-identified case this is the Anderson–Rubin test of $\beta = 0$, and it is valid
regardless of instrument strength — so it distinguishes weak-instrument noise from a genuine
direct effect of $Z$ on $y$.

**The support check** asks whether the implied effect is even arithmetically possible given
the distribution of the endogenous regressor.

Recall $\beta_{IV}$ in the just-identified case is simply the reduced form divided by the
first stage, so a small direct effect combined with a weak first stage produces an
arbitrarily large ratio.

In [6]:
# Reduced form — the Anderson-Rubin test of beta = 0 in the just-identified case,
# valid regardless of instrument strength
rf = pf.feols(f"{Y} ~ Z | {fe_iv}", data=d, vcov={"CRV1": PARENT})
print(rf.tidy())

# Is the implied effect even inside the support of the data?
print(d.loc[d.log_net > 0, "log_net"].describe())
print(f"\nimplied change over the interquartile range of log_net: "
      f"{-1.1398 * (d.loc[d.log_net>0,'log_net'].quantile(.75) - d.loc[d.log_net>0,'log_net'].quantile(.25)):.3f}")

             Estimate  Std. Error   t value  Pr(>|t|)      2.5%     97.5%
Coefficient                                                              
Z           -0.037941    0.011203 -3.386816  0.000857 -0.060036 -0.015846
count    6727.000000
mean        1.833340
std         1.226598
min         0.693147
25%         0.693147
50%         1.386294
75%         2.484907
max         7.013915
Name: log_net, dtype: float64

implied change over the interquartile range of log_net: -2.042


c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


Both checks point the same way, and the direction rules out the benign explanation.

The reduced form is **−0.038 (0.011), p < 0.001**. That is an Anderson–Rubin rejection of
$\beta = 0$, valid at any instrument strength, so the problem is not weak-instrument noise:
$Z$ genuinely predicts lower affiliate patenting.

The arithmetic then explains the second stage exactly. $-0.0379 / 0.0333 = -1.14$. A modest
direct effect divided by a first stage of 0.033 produces the implausible ratio. And the
support check confirms it is implausible rather than merely large: across the interquartile
range of `log_net` among exposed cells (0.69 to 2.48) the implied change is −2.04, or 204
percentage points of a bounded probability.

Strong reduced form, weak first stage, infeasible ratio — the signature of a **failed
exclusion restriction**. $Z$ is reaching the outcome through a channel other than network
stock.

### Naming the channel: is the instrument just a trend?

"The exclusion restriction fails" is a weak claim on its own. This test identifies the
specific channel.

Look at how $Z$ is built. Within a firm-subclass pair, $w_{is}$ is fixed — it is a
pre-2015 share, computed once and never updated. $G_{st}$ is a *cumulative* count, so it
only rises. Their product therefore climbs monotonically over the window, at a rate
proportional to the share:

$$Z_{ist} = \log\!\left(1 + w_{is}G_{st}\right) \quad\text{with } w_{is}\ \text{fixed and } G_{st}\ \text{increasing}$$

That is a **pair-specific time trend**. And the fixed effects cannot remove it: $\delta_{is}$
absorbs pair *levels*, $\gamma_{st}$ absorbs what is common to a subclass-year, but a
trend whose slope varies across pairs within a subclass survives all three.

Why that would produce a negative reduced form without any spillover: pairs where the
group's pre-2015 knowledge was most concentrated are plausibly its mature technologies, and
own patenting in a mature technology drifts down over a ten-year window. High share → steep
$Z$ → declining $y$, with nothing flowing between firms.

The test replaces the shift with a bare linear trend, keeping only the share:

$$Z^{trend}_{ist} = w_{is} \times (t - 2015)$$

This has no shift-share content at all — no national patenting data enters it. If
$Z^{trend}$ alone reproduces the negative reduced form, the instrument's explanatory power
is coming from its trend component rather than from technology-specific shifts. The second
regression puts both in together: if $Z$ loses its coefficient once the trend is controlled,
that is conclusive.

In [7]:
# A pure trend scaled by the share — no shift-share content at all
d["Ztrend"] = d["w"] * (d[YEAR] - 2015)

rf_t = pf.feols(f"{Y} ~ Ztrend | {fe_iv}", data=d, vcov={"CRV1": PARENT})
print(rf_t.tidy(), "\n")

# Does Z survive once the trend is controlled for?
rf_both = pf.feols(f"{Y} ~ Z + Ztrend | {fe_iv}", data=d, vcov={"CRV1": PARENT})
print(rf_both.tidy())

             Estimate  Std. Error   t value  Pr(>|t|)      2.5%     97.5%
Coefficient                                                              
Ztrend      -0.004169    0.004038 -1.032554  0.303105 -0.012132  0.003794 

             Estimate  Std. Error  t value  Pr(>|t|)      2.5%     97.5%
Coefficient                                                             
Z           -0.064839    0.016029 -4.04520  0.000076 -0.096452 -0.033225
Ztrend       0.020931    0.006525  3.20787  0.001566  0.008062  0.033800


c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


In [8]:
print(d[["Z", "Ztrend"]].corr())

               Z    Ztrend
Z       1.000000  0.767196
Ztrend  0.767196  1.000000


The trend hypothesis is refuted. $Z^{trend}$ on its own is −0.004 (0.004), p = 0.30 —
nothing. The instrument is not a disguised time trend, and with both regressors included
$Z$ becomes *stronger* (−0.065) while the trend flips positive. $Z$'s explanatory power
comes from the genuine shift component: the subclass-specific pattern of national patenting
growth, not its monotone drift.

The two regressors correlate at 0.77, high enough that the joint coefficients are partially
partialling against each other and should not be leaned on individually. The
single-regressor results are the reliable ones, and they say the trend channel is not the
explanation.

This does not rescue the instrument. The arithmetic is unchanged: a reduced form of −0.038
cannot run through a first stage of 0.033 and yield a coefficient inside the support of a
bounded probability. An instrument that moves the endogenous variable that little cannot
legitimately produce an outcome effect that large through that channel — something else is
carrying it.

A plausible candidate follows from the construction. $G_{st}$ counts *other groups'*
cumulative patenting in the technology, which is a measure of how crowded the field is,
scaled by the group's concentration in it. Firms in crowded technologies where their group
is concentrated patent less. That is competition acting directly on the outcome, not
knowledge flowing through the network — precisely the kind of channel the exclusion
restriction rules out. It is a conjecture, not a tested claim, but it is consistent with
every number above.

### Testing the crowding conjecture

If the exclusion violation runs through competition, the shift should predict the outcome on
its own, unscaled by the share. $G_{st}$ counts other groups' cumulative patenting in the
technology — a measure of how crowded the field is — and crowding plausibly suppresses a
firm's own filing directly, with no network involved.

A negative coefficient here supports the conjecture and gives the violated channel a name. A
flat one means $Z$'s effect works through the interaction with the share specifically, which
is a different and less obvious story.

In [9]:
d["logG"] = np.log1p(d["G_loo"])
pf.feols(f"{Y} ~ logG | {fe_iv}", data=d, vcov={"CRV1": PARENT}).tidy()

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
logG,-0.309622,0.03946,-7.846482,2.859935e-13,-0.38745,-0.231794


−0.310 (0.039), t = −7.85. Far too strong for a competition story, and the algebra explains
why.

`sub_year` fixed effects absorb anything that is a function of subclass and year alone —
which is exactly $G_{st}$, the total. So within a subclass-year the only thing that varies
in $G^{(-g)}_{st} = G_{st} - G^{own}_{gst}$ is the group's own cumulative stock. The
coefficient of −0.310 on $\log G^{(-g)}$ is therefore +0.310 on the group's own stock, which
includes firm $i$'s own patents.

This is the firm's past patenting predicting its present patenting. Mechanical, not
economic — and the same contamination sits inside $Z$, since $Z = \log(1 + w_{is}G^{(-g)}_{st})$.
That, not competition, is what drives the strong negative reduced form.

The crowding conjecture is therefore not supported; something more basic is going on. The
next cell confirms it directly by putting the group's own stock on the right-hand side.

In [10]:
d["logGown"] = np.log1p(d["G_own"])
pf.feols(f"{Y} ~ logGown | {fe_iv}", data=d, vcov={"CRV1": PARENT}).tidy()

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
logGown,0.196629,0.010599,18.552016,0.0,0.175725,0.217534


Confirmed: +0.197 (0.011), t = 18.6. The sign flip from the −0.310 above is mechanical —
after `sub_year` absorbs the total, the leave-one-out term varies only through $G^{own}$, so
the two variables move inversely by construction and these are the same fact stated twice.
(The magnitudes differ because the log transform gives the two variables different scales;
only the sign relationship is implied.) And $G^{own}$ contains firm $i$'s own patents, so
the coefficient is the firm's past patenting predicting its present patenting.

That identifies the failure precisely, and it is structural rather than empirical:

**A shift-share instrument is incompatible with subclass × year fixed effects.** The fixed
effects absorb the shift — that is their purpose — and what survives is the leave-one-out
correction, which is a function of the firm's own group. The instrument is then tied to the
outcome by construction, whatever the economics.

This resolves the tension anticipated but not settled in the original project plan: the
shares are absorbed by the pair FE, the shifts by the subclass-year FE, and nothing is left
for a shift-share to identify. Dropping the subclass × year control would restore the shift,
but then the instrument's cross-sectional variation is the sorting documented in notebook 6
— which is why that specification was rejected at the outset. The two requirements cannot be
met at once in this design.

## Summary

The Bartik instrument does not work in this design, and the reason is structural rather than
a matter of insufficient data.

**The design.** The instrument pairs a share — the network's pre-2015 composition across
technologies, $w_{is}$ — with a shift — cumulative national patenting in the technology,
$G_{st}$, computed leave-one-out so the firm's own group never enters its own shift. It was
applied to the pair-FE specification rather than the between-technology one, because
identification in a shift-share comes from the shares, and notebook 6 showed the network's
pre-period composition is strongly correlated with the affiliate's own patenting. Only the
pair FE remove that correlation.

**It looked viable.** The instrument varies in 670 pairs, 278 of which also see `log_net`
move — about 71% of the usable variation. The first stage is positive and passes the
conventional threshold: $\pi = 0.033$ (0.009), $F = 12.8$ under parent-group clustering.

**The second stage is impossible.** $\beta_{IV} = -1.14$ (0.50) against an OLS of $-0.011$
(0.025). The outcome is a probability; across the interquartile range of `log_net` among
exposed cells the estimate implies a change of −2.04, or 204 percentage points. A reduced
form of −0.038 divided by a first stage of 0.033 produces the ratio — a modest direct effect
scaled up by a weak first stage.

**The reduced form is not noise.** At −0.038 (0.011), p < 0.001, it is an Anderson–Rubin
rejection valid at any instrument strength. $Z$ genuinely predicts lower affiliate
patenting, so the problem is the exclusion restriction, not precision.

**Why.** Not a time trend — the share-scaled trend alone is flat, −0.004 (0.004). The
mechanism is arithmetic. `sub_year` fixed effects absorb $G_{st}$, since it is constant
within a subclass-year by construction. What survives in $G^{(-g)}_{st} = G_{st} -
G^{own}_{gst}$ is only the group's own cumulative stock — which contains firm $i$'s own
patents. Confirmed directly: $\log G^{(-g)}$ gives −0.310 (0.039) and $\log G^{own}$ gives +0.197
(0.011). The opposite signs are mechanical — within a subclass-year the two variables move
inversely by construction — so these are the same fact stated twice.

**The structural conclusion.** A shift-share instrument is incompatible with subclass × year
fixed effects. The fixed effects absorb the shift — that is what they are for — leaving only
the leave-one-out correction, which is a function of the firm's own group. Dropping the
subclass × year control restores the shift, but then the instrument's identifying variation
is the sorting documented in notebook 6, which is why that specification was rejected at the
outset. The two requirements cannot be met simultaneously here.

This answers the open item carried since the original project plan, which anticipated the
tension — shares absorbed by the pair FE, shifts by the technology-year FE — without
resolving it. The resolution is that there is nothing left for a shift-share to identify.

**What would be needed.** A shift constructed from data outside the sample entirely, so that
the leave-one-out correction is not a function of the estimation sample's own firms. That
requires a new data source, or the foreign parent-portfolio route ruled out on scope grounds
at the start of notebook 6. Both are limitations to state, not next steps.